In [ ]:
import cv2
import numpy as np
import os
from ultralytics import YOLO

# === CONFIGURATION ===
save_h_path = r'initial_homography.npy'
output_path = r"C:\Users\Anwari Ahmed\Desktop\Internship\AR_Mandible\tracked_overlay_line.avi"
model_path = r"C:\Users\Anwari Ahmed\Desktop\Internship\AR_Mandible\runs\detect\yolo_custom_model_12\weights\best.pt"
model = YOLO(model_path)

class_names = {
    0: "R Canine", 1: "R Lateral", 2: "R Central",
    3: "L Central", 4: "L Lateral", 5: "Extra",
    6: "L Canine", 7: "Molar"
}

def prompt_ct_path():
    while True:
        ct_path = input("Enter full path to the new CT scan image: ").strip()
        if os.path.exists(ct_path):
            img = cv2.imread(ct_path)
            if img is not None:
                return ct_path, img
            print("[ERROR] Unable to read image file. Try again.")
        else:
            print("[ERROR] Invalid path. Try again.")

def capture_frame_from_camera(cap):
    print("Press 'c' to capture a frame.")
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame.")
            continue
        cv2.imshow("Live Feed", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('c'):
            cv2.destroyWindow("Live Feed")
            return frame.copy()
        elif key == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            exit()

def detect_yolo_landmarks(frame):
    results = model(frame)[0]
    centers = {name: None for name in class_names.values()}
    for box in results.boxes:
        cls_id = int(box.cls.item())
        label = class_names.get(cls_id)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
        centers[label] = (cx, cy)

    points = []
    for label, pt in centers.items():
        if pt is not None:
            cv2.circle(frame, pt, 6, (0, 0, 255), -1)
            cv2.putText(frame, label, (pt[0] + 5, pt[1] - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            points.append(pt)
        else:
            print(f"[WARNING] {label} not detected.")
    return np.array(points, dtype=np.float32), frame

def select_points_with_display(image, max_points=10, window_name="Select Points"):
    points = []
    display = image.copy()

    def click_event(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN and len(points) < max_points:
            points.append((x, y))
            cv2.circle(display, (x, y), 5, (0, 255, 0), -1)
            cv2.imshow(window_name, display)

    cv2.imshow(window_name, display)
    cv2.setMouseCallback(window_name, click_event)
    print(f"[INFO] Click up to {max_points} points in '{window_name}'. Press any key when done.")
    cv2.waitKey(0)
    cv2.destroyWindow(window_name)
    return points

def draw_line_on_overlay(overlay):
    line_points = []
    clone = overlay.copy()

    def draw_line_event(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN and len(line_points) < 10:
            line_points.append([x, y])
            cv2.circle(param, (x, y), 4, (0, 255, 255), -1)
            if len(line_points) > 1:
                cv2.polylines(param, [np.array(line_points)], isClosed=False, color=(0, 255, 0), thickness=2)
            cv2.imshow("Draw Line to Track", param)

    cv2.namedWindow("Draw Line to Track")
    cv2.setMouseCallback("Draw Line to Track", draw_line_event, clone)
    print("Draw 4–10 points to define a line. Press ENTER to continue.")

    while True:
        temp = clone.copy()
        if len(line_points) >= 2:
            cv2.polylines(temp, [np.array(line_points)], isClosed=False, color=(0, 255, 0), thickness=2)
        cv2.imshow("Draw Line to Track", temp)
        key = cv2.waitKey(10) & 0xFF
        if key == 13 or key == 10:
            break
        elif key == 27:
            print("Canceled. Exiting.")
            cv2.destroyAllWindows()
            return None

    cv2.destroyWindow("Draw Line to Track")
    return line_points

def run_tracking_loop(cap, cap_frame, warped_ct):
    mask = cv2.cvtColor(warped_ct, cv2.COLOR_BGR2GRAY) > 10
    mask_3ch = cv2.merge([mask.astype(np.uint8)] * 3)

    alpha = 0.3
    overlay = cap_frame.copy()
    overlay[mask_3ch == 1] = cv2.addWeighted(cap_frame, 1 - alpha, warped_ct, alpha, 0)[mask_3ch == 1]

    line_points = draw_line_on_overlay(overlay)
    if line_points is None or len(line_points) < 2:
        return False

    print("[INFO] Line drawn. Now align with the real object. Press 't' to start tracking.")

    while True:
        ret, live_frame = cap.read()
        if not ret:
            continue
        guide_frame = live_frame.copy()
        cv2.polylines(guide_frame, [np.array(line_points)], isClosed=False, color=(0, 255, 0), thickness=2)
        cv2.imshow("Guide - Press 't' to start tracking", guide_frame)

        key = cv2.waitKey(10) & 0xFF
        if key == ord('t'):
            cap_frame = live_frame.copy()
            break
        elif key == ord('q') or key == 27:
            print("Exiting.")
            return False

    cv2.destroyWindow("Guide - Press 't' to start tracking")

    prev_gray = cv2.cvtColor(cap_frame, cv2.COLOR_BGR2GRAY)
    points_to_track = np.array(line_points, dtype=np.float32).reshape(-1, 1, 2)

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 10
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    print("[INFO] Tracking started. Press 'e' to pause/resume, 'r' to re-register or 'q' to quit.")
    paused = False

    while True:
        ret, curr_frame = cap.read()
        if not ret:
            break

        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)

        if not paused:
            new_points, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, points_to_track, None)
            if new_points is not None and status is not None:
                valid_new = new_points[status.flatten() == 1]
                if len(valid_new) >= 2:
                    updated_points = valid_new.reshape(-1, 2).astype(np.int32)
                    cv2.polylines(curr_frame, [updated_points], isClosed=False, color=(0, 255, 0), thickness=2)
                    points_to_track = valid_new.reshape(-1, 1, 2)
                else:
                    updated_points = points_to_track.reshape(-1, 2).astype(np.int32)
                    cv2.polylines(curr_frame, [updated_points], isClosed=False, color=(0, 255, 255), thickness=2)
            prev_gray = curr_gray.copy()
        else:
            static_points = points_to_track.reshape(-1, 2).astype(np.int32)
            cv2.polylines(curr_frame, [static_points], isClosed=False, color=(255, 0, 0), thickness=2)

        cv2.imshow("Tracked Curve", curr_frame)
        video_writer.write(curr_frame)

        key = cv2.waitKey(10) & 0xFF
        if key == ord('r'):
            print("Re-registration triggered.")
            video_writer.release()
            return True
        elif key == ord('q') or key == 27:
            print("Exiting.")
            video_writer.release()
            return False
        elif key == ord('e'):
            paused = not paused
            print(f"[INFO] Tracking {'paused' if paused else 'resumed'}.")

    video_writer.release()
    return False

# === MAIN LOOP ===
cap = cv2.VideoCapture(r'C:\Users\Anwari Ahmed\Desktop\Internship\AR_Mandible\edited.mp4')
if not cap.isOpened():
    raise IOError("Cannot open webcam")

while True:
    cap_frame = capture_frame_from_camera(cap)
    pts2, annotated_frame = detect_yolo_landmarks(cap_frame)

    if len(pts2) < 4:
        print("[ERROR] Not enough landmarks detected. Try again.")
        continue

    # Show detected landmarks immediately
    cv2.imshow("Captured Points (Video)", annotated_frame)
    cv2.waitKey(1)

    ct_image_path, initial_ct = prompt_ct_path()

    while True:
        scale = 0.5
        resized_ct = cv2.resize(initial_ct.copy(), (int(initial_ct.shape[1] * scale), int(initial_ct.shape[0] * scale)))

        print("[INFO] Select corresponding points on the CT image.")
        pts1_scaled = select_points_with_display(resized_ct, max_points=len(pts2), window_name="CT Image")

        # Close windows after selection
        
        cv2.destroyWindow("Captured Points (Video)")

        if len(pts1_scaled) != len(pts2):
            print("[ERROR] Mismatch in number of points. Retry.")
            continue

        pts1 = np.array(pts1_scaled, dtype=np.float32) / scale
        pts2 = np.array(pts2, dtype=np.float32)
        break

    H_ct_to_0, _ = cv2.findHomography(pts1, pts2)
    np.save(save_h_path, H_ct_to_0)

    warped_ct = cv2.warpPerspective(initial_ct, H_ct_to_0, (cap_frame.shape[1], cap_frame.shape[0]))

    rerun = run_tracking_loop(cap, cap_frame, warped_ct)
    if not rerun:
        break


cap.release()
cv2.destroyAllWindows()
print(f"[INFO] Video saved to: {output_path}")
